## Resources

This notebook implements **style-aligned batch image generation** based on the method described at [https://github.com/google/style-aligned/](https://github.com/google/style-aligned/).

Implementation utilizes the NVIDIA SANA Sprint 0.6b 2-step model.

In [ ]:
from diffusers import SanaSprintPipeline
import torch,gc,imageio,glob
from PIL import Image

pipe = SanaSprintPipeline.from_pretrained(
    "Efficient-Large-Model/SANA_Sprint_0.6B_1024px_diffusers",
    torch_dtype=torch.float16,
)
pipe.to("cuda:0")
pipe.vae.to(torch.float16)
pipe.vae.enable_tiling(tile_sample_min_width=512, tile_sample_min_height=512)

In [ ]:
%%writefile style_aligned_sana.py

"""
StyleAligned for NVIDIA Sana Sprint (0.6B, 2-step distilled, linear-attention DiT)
====================================================================================

Reference: https://github.com/google/style-aligned  ("Style Aligned Image Generation via Shared Attention")


Usage
-----
    from diffusers import SanaSprintPipeline
    from style_aligned_sana import apply_style_aligned, StyleAlignedArgs

    pipe = SanaSprintPipeline.from_pretrained(
        "Efficient-Large-Model/Sana_Sprint_0.6B_1024px_diffusers",
        torch_dtype=torch.bfloat16,
    ).to("cuda")

    handle = apply_style_aligned(pipe, StyleAlignedArgs())

    prompts = [
        "a photo of a lighthouse, watercolor painting",   # index 0 = style reference
        "a photo of a cat, watercolor painting",
        "a photo of a mountain, watercolor painting",
        "a photo of a car, watercolor painting",
    ]

    images = pipe(
        prompt=prompts,
        num_inference_steps=2,          # Sana Sprint's native step count
        generator=torch.Generator("cuda").manual_seed(0),
    ).images

    handle.remove()  # restore original attention processors when done
"""

from __future__ import annotations
import dataclasses
from typing import Optional
import numpy as np
import torch
import torch.nn.functional as F


def adain(feat: torch.Tensor, ref: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    feat: (B, H, N, D) queries/keys/values for the whole batch
    ref:  (1, H, N_ref, D) same tensor sliced at the reference index (keepdim)
    Normalizes `feat`'s per-(batch,head,channel) statistics over the token
    dimension to match `ref`'s statistics.
    """
    feat_mean = feat.mean(dim=2, keepdim=True)
    feat_std = feat.std(dim=2, keepdim=True) + eps
    ref_mean = ref.mean(dim=2, keepdim=True)
    ref_std = ref.std(dim=2, keepdim=True) + eps
    return (feat - feat_mean) / feat_std * ref_std + ref_mean


@dataclasses.dataclass
class StyleAlignedArgs:

    adain_queries: bool = True
    adain_keys: bool = True
    adain_values: bool = False
    # Whether to concatenate the reference's K/V into every sample's own K/V
    # this is the actual "shared attention" step; AdaIN alone is a weaker,
    # Down-weight the reference's contribution to the shared KV state if you want less aggressive style transfer (1.0 = full weight, matches paper).
    shared_score_scale: float = 1.0
    #Index within the batch that acts as the style reference
    reference_index: int = 0
    layer_indices: Optional[set] = None
    eps: float = 1e-6


class SharedLinearAttnProcessor:
    """
    Drop-in replacement for `SanaLinearAttnProcessor2_0` on `attn1` (self-attention)
    that performs StyleAligned's AdaIN + shared-attention across the batch,
    reimplemented for Sana's ReLU-kernel linear attention.

    """

    def __init__(self, args: StyleAlignedArgs):
        self.args = args

    def __call__(
        self,
        attn,
        hidden_states: torch.Tensor,
        encoder_hidden_states: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> torch.Tensor:
        a = self.args
        batch_size, seq_len, _ = hidden_states.shape

        query = attn.to_q(hidden_states)
        key = attn.to_k(hidden_states)
        value = attn.to_v(hidden_states)

        if attn.norm_q is not None:
            query = attn.norm_q(query)
        if attn.norm_k is not None:
            key = attn.norm_k(key)

        heads = attn.heads
        head_dim = query.shape[-1] // heads

        def split_heads(x):
            return x.view(batch_size, -1, heads, head_dim).transpose(1, 2)  # (B, H, N, D)

        orig_dtype = query.dtype


        q, k, v = split_heads(query), split_heads(key), split_heads(value)
        q, k, v = q.float(), k.float(), v.float()

        ref = a.reference_index

        # --- Apply the ReLU kernel feature map FIRST, before AdaIN. ---
        # StyleAligned's AdaIN was designed for softmax attention, which is
        # shift-tolerant. If AdaIN runs on raw Q/K and the reference happens
        # to have a negative mean, it drags every other sample's Q/K negative
        # too -- and since Sana's linear attention kernel is relu(Q)/relu(K),
        # that gets clipped to all-zero, collapsing attention entirely
        # (symptom: valid-looking floats, no NaNs, but a flat black decode).
        # Doing AdaIN on the already-nonnegative relu(Q)/relu(K) avoids this,
        # and we re-clamp afterward since AdaIN's rescale can still dip
        # slightly negative near the low end of the distribution.
        q = F.relu(q)
        k = F.relu(k)

        # --- AdaIN: pull each sample's Q/K statistics toward the reference's ---
        if a.adain_queries:
            q = adain(q, q[ref : ref + 1], a.eps).clamp_min(0.0)
        if a.adain_keys:
            k = adain(k, k[ref : ref + 1], a.eps).clamp_min(0.0)
        if a.adain_values:
            v = adain(v, v[ref : ref + 1], a.eps)

        # --- Shared attention: fold the reference's K/V into every sample's own
        if a.share_attention:
            k_ref = k[ref : ref + 1].expand(batch_size, -1, -1, -1) * a.shared_score_scale
            v_ref = v[ref : ref + 1].expand(batch_size, -1, -1, -1)
            k = torch.cat([k, k_ref], dim=2)
            v = torch.cat([v, v_ref], dim=2)

        kv_state = torch.einsum("bhnd,bhne->bhde", k, v)
        k_sum = k.sum(dim=2)

        out = torch.einsum("bhnd,bhde->bhne", q, kv_state)
        denom = torch.einsum("bhnd,bhd->bhn", q, k_sum).unsqueeze(-1)
        denom = denom.clamp_min(a.eps)
        out = out / denom

        out = out.transpose(1, 2).reshape(batch_size, -1, heads * head_dim)
        out = torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
        out = out.to(orig_dtype)

        out = attn.to_out[0](out)
        out = attn.to_out[1](out)
        out = out / attn.rescale_output_factor
        return out


class StyleAlignedHandle:

    def __init__(self, transformer, original_processors: dict):
        self._transformer = transformer
        self._original = original_processors

    def remove(self):
        self._transformer.set_attn_processor(self._original)


def apply_style_aligned(pipe, args: StyleAlignedArgs = StyleAlignedArgs()) -> StyleAlignedHandle:
    """
    Swaps `attn1` (self-attention) processors on every SanaTransformerBlock in
    `pipe.transformer` for SharedLinearAttnProcessor. Cross-attention (attn2,
    text conditioning) is left untouched -- StyleAligned only touches
    self-attention.

    Call `.remove()` on the returned handle to restore original behavior.
    """
    transformer = pipe.transformer
    original_processors = transformer.attn_processors  # dict: name -> processor

    new_processors = {}
    for name, proc in original_processors.items():
        if name.endswith("attn1.processor"):
            if args.layer_indices is not None:
                # names look like "transformer_blocks.{i}.attn1.processor"
                idx = int(name.split(".")[1])
                if idx not in args.layer_indices:
                    new_processors[name] = proc
                    continue
            new_processors[name] = SharedLinearAttnProcessor(args)
        else:
            new_processors[name] = proc

    transformer.set_attn_processor(new_processors)
    return StyleAlignedHandle(transformer, original_processors)




@torch.no_grad()
def encode_reference_image(pipe, image, generator=None):

    img = torch.from_numpy(np.array(image)).float() / 127.5 - 1.0
    img = img.permute(2, 0, 1).unsqueeze(0).to(pipe.device, dtype=pipe.vae.dtype)
    latent = pipe.vae.encode(img).latent
    latent = latent * pipe.vae.config.scaling_factor
    return latent

In [ ]:
from style_aligned_sana import apply_style_aligned, StyleAlignedArgs

args = StyleAlignedArgs(
    adain_queries=True,
    adain_keys=True,
    adain_values=True,       # turn this ON — color/texture consistency usually needs it
    share_attention=True,
    shared_score_scale=1.0,
)
handle = apply_style_aligned(pipe, args)

In [ ]:
prompts = [
    "a toy train. macro photo. 3d game asset.",   # reference (index 0)
    "a toy airplane. macro photo. 3d game asset.",
    "a toy bicycle. macro photo. 3d game asset.",
    "a toy car. macro photo. 3d game asset.",
    "a toy boat. macro photo. 3d game asset.",
]


images = pipe(
    prompt=prompts,
    num_inference_steps=2,
).images

def grid(imgs, cols):
    w, h = imgs[0].size
    rows = (len(imgs) + cols - 1) // cols
    canvas = Image.new("RGB", (w * cols, h * rows), "white")
    for i, im in enumerate(imgs):
        canvas.paste(im, ((i % cols) * w, (i // cols) * h))
    return canvas

grid(images, cols=len(images))